# <font color="#76b900">**Notebook 3:** Fine-tuning an LLM with LoRA on DGX Spark</font>

### Overview

In Notebook 2 we improved results with zero-shot inference and in-context learning
(ICL). Those methods are bounded — ICL doesn't *permanently* improve the model and
costs context on every request. This notebook uses **LoRA (Low-Rank Adaptation)** to
permanently adapt a model to the legal-title task, then compares the fine-tuned model
against the base.

### What is LoRA?

LoRA freezes the original model weights and trains small "rank decomposition"
matrices on selected layers — often < 1% of the parameters — giving most of the
benefit of full fine-tuning at a fraction of the cost.

### Learning Objectives

- Prepare legal-domain data for LoRA fine-tuning
- Launch a LoRA training job with **NeMo AutoModel** on the Spark's GPU
- Inspect the adapter and training curves
- Compare the fine-tuned model against the base model

> ### <font color="#76b900">Running on DGX Spark</font>
> Training runs in the **NeMo AutoModel** container
> (`nvcr.io/nvidia/nemo-automodel:26.06`), whose PyTorch supports the GB10's `sm_121`.
>
> **Prerequisites:** [`00-NIM-Setup-DGX-Spark.ipynb`](00-NIM-Setup-DGX-Spark.ipynb),
> [`00c-NeMo-AutoModel-Setup-DGX-Spark.ipynb`](00c-NeMo-AutoModel-Setup-DGX-Spark.ipynb),
> and the `data/law-qa-*.jsonl` splits (already in this repo).
>
> ⚠️ **This notebook stops the NIM during training** — fine-tuning needs the GPU
> memory the running NIM reserves (~84 GB). It restarts the NIM afterwards.

<br><hr>

### Which model we fine-tune, and why not Nemotron

Notebooks 01–02 evaluated the **Nemotron Nano 9B** NIM. This notebook fine-tunes
**Llama-3.2-3B-Instruct** instead, because on this container + Spark neither Nemotron
option is currently trainable:

| Model | Status on the Spark |
|---|---|
| Nemotron-Nano-9B-v2 (Mamba hybrid) | Needs a compiled `mamba_ssm` extension — the container ships it x86-only, and it fails to build on the Spark's CUDA-13 / sm_121 toolchain. |
| Nemotron-3 Nano 30B-A3B (Mamba-free) | Loads natively, but OOMs — weight conversion transiently needs ~116 GB of the 128 GB budget. |
| **Llama-3.2-3B-Instruct** (standard transformer) | Trains cleanly in minutes, ~8 GB peak. ✅ |

The LoRA workflow below is identical for any supported model — swap
`pretrained_model_name_or_path` in the recipe to retarget it once Nemotron support
catches up.

> We use the ungated mirror **`unsloth/Llama-3.2-3B-Instruct`** so no HF license
> gate/token is needed. With an HF token that has Llama access, use the canonical
> `meta-llama/Llama-3.2-3B-Instruct`.

In [ ]:
import os, json, yaml, time, requests, subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# === DGX Spark configuration =================================================
NIM_HOST       = "http://localhost:8000"
NIM_CONTAINER  = "nemotron-nano"
BASE_MODEL     = "unsloth/Llama-3.2-3B-Instruct"
AUTOMODEL_IMAGE = "nvcr.io/nvidia/nemo-automodel:26.06"

os.environ.update(
    NIM_HOST=NIM_HOST, NIM_CONTAINER=NIM_CONTAINER, BASE_MODEL=BASE_MODEL
)
os.makedirs("automodel_recipes", exist_ok=True)
print("NIM_HOST       =", NIM_HOST)
print("NIM_CONTAINER  =", NIM_CONTAINER)
print("BASE_MODEL     =", BASE_MODEL)
print("AutoModel image=", AUTOMODEL_IMAGE)

<br><hr>

## Step 1 — Inspect the training data

We reuse the **same legal-title splits as Notebook 2** — `law-qa-train.jsonl`,
`law-qa-val.jsonl`, `law-qa-test.jsonl` — in the `{"prompt": <question body>,
"completion": <title>}` format. AutoModel reads them directly with
`ColumnMappedTextInstructionDataset` (`question → prompt`, `answer → completion`,
loss masked to the title only).

In [ ]:
DATA_DIR = "data"
SPLITS = {name: os.path.join(DATA_DIR, f"law-qa-{name}.jsonl")
          for name in ["train", "val", "test"]}

for name, path in SPLITS.items():
    rows = [json.loads(l) for l in open(path)]
    print(f"{name:6s}: {len(rows):4d} rows | fields = {list(rows[0].keys())}")

ex = json.loads(open(SPLITS["train"]).readline())
print("\nExample pair:\n  prompt    :", ex["prompt"][:100], "...\n  completion:", ex["completion"])

<br><hr>

## Step 2 — Write the training recipe and launcher

NeMo AutoModel is configured with a YAML recipe. Key settings: LoRA `dim: 8`,
`alpha: 32`, `target_modules: '*_proj'`, single-GPU `fsdp2`, 3 epochs, `lr 1e-4`.

In [ ]:
%%writefile automodel_recipes/law_qa_llama_lora.yaml
# LoRA (PEFT) fine-tuning of Llama-3.2-3B-Instruct on the legal-title task,
# on a single DGX Spark GPU.
recipe: TrainFinetuneRecipeForNextTokenPrediction

step_scheduler:
  global_batch_size: 8
  local_batch_size: 1
  num_epochs: 3
  max_steps: 300
  val_every_steps: 50
  ckpt_every_steps: 50

dist_env:
  backend: nccl
  timeout_minutes: 20

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true

model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: unsloth/Llama-3.2-3B-Instruct
  torch_dtype: bf16

checkpoint:
  enabled: true
  checkpoint_dir: /workspace/checkpoints
  model_save_format: safetensors
  save_consolidated: true

peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  target_modules: '*_proj'
  dim: 8
  alpha: 32
  dropout: 0.1
  use_triton: true

distributed:
  strategy: fsdp2
  dp_size: none
  tp_size: 1
  cp_size: 1
  sequence_parallel: false

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: /workspace/data/law-qa-train.jsonl
  split: train
  column_mapping:
    question: prompt
    answer: completion
  seq_length: 2048
  answer_only_loss_mask: true
  padding: do_not_pad
  truncation: longest_first

validation_dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: /workspace/data/law-qa-val.jsonl
  split: validation
  column_mapping:
    question: prompt
    answer: completion
  seq_length: 2048
  answer_only_loss_mask: true
  padding: do_not_pad
  truncation: longest_first

packed_sequence:
  packed_sequence_size: 0

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  shuffle: true

validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater

optimizer:
  _target_: torch.optim.Adam
  lr: 1.0e-4
  weight_decay: 0.01
  betas: [0.9, 0.999]
  eps: 1.0e-8

We launch training through a small **wrapper** rather than AutoModel's `finetune.py`
directly. The `nemo-automodel:26.06` container has one transformers-5.x
incompatibility — AutoModel's `initialize_weights()` calls `LlamaRMSNorm.reset_parameters()`,
which transformers 5.x removed — so the wrapper re-adds it, then runs the standard
entrypoint unchanged.

In [ ]:
%%writefile automodel_recipes/spark_finetune.py
"""DGX Spark launcher for NeMo AutoModel 26.06.

Patches one transformers-5.x incompatibility: AutoModel's initialize_weights()
calls LlamaRMSNorm.reset_parameters(), which transformers 5.x removed. We re-add it
(RMSNorm init = weights set to 1), then run the stock finetune entrypoint unchanged.
"""
import torch, runpy
from transformers.models.llama.modeling_llama import LlamaRMSNorm

def _reset(self):
    with torch.no_grad():
        self.weight.fill_(1.0)

if not hasattr(LlamaRMSNorm, "reset_parameters"):
    LlamaRMSNorm.reset_parameters = _reset
    print("[spark-patch] added LlamaRMSNorm.reset_parameters")

runpy.run_path("/opt/Automodel/examples/llm_finetune/finetune.py", run_name="__main__")

In [ ]:
cfg = yaml.safe_load(open("automodel_recipes/law_qa_llama_lora.yaml"))
recipe_model = cfg["model"]["pretrained_model_name_or_path"]
assert recipe_model == BASE_MODEL, f"recipe trains {recipe_model} but BASE_MODEL is {BASE_MODEL}"

print("Recipe OK — model:", recipe_model)
print("LoRA dim:", cfg["peft"]["dim"], "| target_modules:", cfg["peft"]["target_modules"])
print("Train data:", cfg["dataset"]["path_or_dataset_id"])

<br><hr>

## Step 3 — Free the GPU (stop the NIM)

⚠️ **This stops your running NIM.** Training needs its GPU memory. Notebooks 01/02
won't work again until you restart it in **Step 6**.

In [ ]:
running = subprocess.run(
    ["docker", "ps", "-q", "--filter", f"name=^{NIM_CONTAINER}$"],
    capture_output=True, text=True, check=True,
).stdout.strip()
NIM_WAS_RUNNING = bool(running)
if NIM_WAS_RUNNING:
    subprocess.run(["docker", "stop", NIM_CONTAINER], check=True)
    print(f"Stopped {NIM_CONTAINER} for training.")
else:
    print(f"{NIM_CONTAINER} was already stopped; leaving it stopped after this notebook.")
subprocess.run([
    "nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv"
], check=False)

<br><hr>

## Step 4 — Launch LoRA fine-tuning

We run the launcher inside the container, mounting this repo at `/workspace` and the
HF cache so the model is downloaded once (~6 GB for Llama-3.2-3B). On the GB10 this
is ~1 step/s; 3 epochs over the legal set is a few minutes. Per-step **loss** streams
below (watch it fall).

In [ ]:
repo = str(Path.cwd())
hf_cache = str(Path.home() / ".cache" / "huggingface")
cmd = [
    "docker", "run", "--rm", "--gpus", "all", "--shm-size", "8g",
    "--ulimit", "memlock=-1", "--ulimit", "stack=67108864",
    "-v", f"{repo}:/workspace", "-w", "/workspace",
    "-v", f"{hf_cache}:/root/.cache/huggingface",
    AUTOMODEL_IMAGE,
    "python3", "/workspace/automodel_recipes/spark_finetune.py",
    "-c", "/workspace/automodel_recipes/law_qa_llama_lora.yaml",
    "--nproc-per-node", "1",
]
print(" ".join(cmd), "\n")
try:
    subprocess.run(cmd, check=True)
except BaseException:
    if NIM_WAS_RUNNING:
        subprocess.run(["docker", "start", NIM_CONTAINER], check=False)
        print(f"Restarted {NIM_CONTAINER} after the interrupted/failed training run.")
    raise

<br><hr>

## Step 5 — Inspect the training curves

AutoModel logs per-step metrics to `checkpoints/training.jsonl` and
`checkpoints/validation.jsonl`, and saves adapters under `checkpoints/` with
`LATEST` and `LOWEST_VAL` symlinks (`LOWEST_VAL` = best validation loss, which we use
for inference).

In [ ]:
train = [json.loads(l) for l in open("checkpoints/training.jsonl")]
val   = [json.loads(l) for l in open("checkpoints/validation.jsonl")]
print(f"Final train loss: {train[-1]['loss']:.4f} | Best val loss: {min(v['val_loss'] for v in val):.4f}")

plt.figure(figsize=(9, 5))
plt.plot([t["step"] for t in train], [t["loss"] for t in train], label="train loss", alpha=0.7)
plt.scatter([v["step"] for v in val], [v["val_loss"] for v in val], color="red", label="val loss", zorder=5)
plt.xlabel("training step"); plt.ylabel("loss"); plt.title("LoRA fine-tuning loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

for link in ["checkpoints/LATEST", "checkpoints/LOWEST_VAL"]:
    if os.path.islink(link):
        print(f"{link} -> {os.readlink(link)}")

if NIM_WAS_RUNNING:
    subprocess.run(["docker", "start", NIM_CONTAINER], check=True)
    deadline = time.monotonic() + 600
    while time.monotonic() < deadline:
        try:
            if requests.get(f"{NIM_HOST}/v1/health/ready", timeout=3).status_code == 200:
                print("NIM ready.")
                break
        except requests.exceptions.RequestException:
            pass
        time.sleep(5)
    else:
        raise TimeoutError(f"NIM did not restart within 10 minutes. Run: docker logs {NIM_CONTAINER}")
else:
    print("NIM was stopped before training, so it remains stopped.")

In [ ]:
if NIM_WAS_RUNNING:
    subprocess.run(["docker", "start", NIM_CONTAINER], check=True)
    deadline = time.monotonic() + 600
    while time.monotonic() < deadline:
        try:
            if requests.get(f"{NIM_HOST}/v1/health/ready", timeout=3).status_code == 200:
                print("NIM ready.")
                break
        except requests.exceptions.RequestException:
            pass
        time.sleep(5)
    else:
        raise TimeoutError(
            f"NIM did not restart within 10 minutes. Run: docker logs {NIM_CONTAINER}"
        )
else:
    print("NIM was stopped before training, so it remains stopped.")

<br><hr>

## Step 7 — Compare base vs fine-tuned

Generate titles for all held-out questions with both the base model and the LoRA
adapter. Both passes use the same model load, prompts, and deterministic decoding.

In [ ]:
test = [json.loads(line) for line in open(SPLITS["test"])]
json.dump([row["prompt"] for row in test],
          open("automodel_recipes/compare_prompts.json", "w"))
refs = [row["completion"] for row in test]
print("Prepared", len(test), "held-out comparison prompts")

In [ ]:
%%writefile automodel_recipes/compare_infer.py
"""Generate base and LoRA titles for every held-out legal question in-container."""
import os, json, glob
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = os.environ["BASE_MODEL"]
candidates = [p for p in ["/workspace/checkpoints/LOWEST_VAL/model",
                           "/workspace/checkpoints/LATEST/model"] if os.path.isdir(p)]
if not candidates:
    candidates = sorted(glob.glob("/workspace/checkpoints/**/model", recursive=True))
if not candidates:
    raise FileNotFoundError("No LoRA adapter found under /workspace/checkpoints")
adapter = candidates[0]
print("Base:", BASE, "| Adapter:", adapter, flush=True)

tokenizer = AutoTokenizer.from_pretrained(adapter)
base = AutoModelForCausalLM.from_pretrained(
    BASE, dtype=torch.bfloat16, device_map="cuda"
).eval()
model = PeftModel.from_pretrained(base, adapter).eval()

SYSTEM = ("You are a headline generator for a legal Q&A forum. Return ONE concise title "
          "(max 15 words) capturing the core legal issue. Output only the title.")
prompts = json.load(open("/workspace/automodel_recipes/compare_prompts.json"))

def generate(active_model, label):
    titles = []
    for index, prompt in enumerate(prompts, 1):
        messages = [{"role": "system", "content": SYSTEM},
                    {"role": "user", "content": prompt}]
        encoded = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        )
        input_ids = encoded["input_ids"].to("cuda")
        with torch.no_grad():
            output = active_model.generate(
                input_ids, max_length=input_ids.shape[1] + 48, do_sample=False
            )
        text = tokenizer.decode(
            output[0][input_ids.shape[1]:], skip_special_tokens=True
        ).strip()
        titles.append(text.splitlines()[0] if text else "")
        if index % 10 == 0 or index == len(prompts):
            print(f"{label}: {index}/{len(prompts)}", flush=True)
    return titles

fine_tuned_titles = generate(model, "LoRA")
with model.disable_adapter():
    base_titles = generate(model, "base")

json.dump({"base": base_titles, "ft": fine_tuned_titles},
          open("/workspace/automodel_recipes/compare_out.json", "w"))

In [ ]:
cmd = [
    "docker", "run", "--rm", "--gpus", "all", "--shm-size", "8g",
    "-v", f"{repo}:/workspace", "-w", "/workspace",
    "-v", f"{hf_cache}:/root/.cache/huggingface",
    "-e", f"BASE_MODEL={BASE_MODEL}",
    AUTOMODEL_IMAGE,
    "python3", "/workspace/automodel_recipes/compare_infer.py",
]
print(" ".join(cmd), "\n")
subprocess.run(cmd, check=True)

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu

with open("automodel_recipes/compare_out.json") as f:
    out = json.load(f)
if not (len(out["base"]) == len(out["ft"]) == len(refs)):
    raise ValueError("Comparison output length does not match the held-out references.")

comparison = pd.DataFrame({
    "question": [row["prompt"][:60] + "..." for row in test],
    "reference": refs,
    "base": out["base"],
    "fine_tuned (LoRA)": out["ft"],
})
pd.set_option("display.max_colwidth", 55)
print(comparison.head(10).to_string(index=False))
print(f"\nShowing 10 of {len(comparison)} held-out rows.")

rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def token_f1(reference, prediction):
    ref_tokens = set(reference.lower().split())
    pred_tokens = set(prediction.lower().split())
    overlap = len(ref_tokens & pred_tokens)
    precision = overlap / len(pred_tokens) if pred_tokens else 0.0
    recall = overlap / len(ref_tokens) if ref_tokens else 0.0
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

def metrics(predictions):
    rouge_scores = [rouge.score(ref, pred) for ref, pred in zip(refs, predictions)]
    return {
        "bleu": sacrebleu.corpus_bleu(predictions, [refs]).score / 100.0,
        "rouge1": sum(score["rouge1"].fmeasure for score in rouge_scores) / len(refs),
        "rougeL": sum(score["rougeL"].fmeasure for score in rouge_scores) / len(refs),
        "token_f1": sum(token_f1(ref, pred) for ref, pred in zip(refs, predictions)) / len(refs),
    }

base_metrics = metrics(out["base"])
lora_metrics = metrics(out["ft"])
metric_table = pd.DataFrame({"base": base_metrics, "fine_tuned (LoRA)": lora_metrics})
metric_table["delta"] = metric_table["fine_tuned (LoRA)"] - metric_table["base"]
print("\nLike-for-like held-out metrics:")
print(metric_table.round(4))

The comparison above is like-for-like: both passes use the same Llama-3.2-3B base,
prompts, decoding settings, and 46 held-out questions. A positive `delta` means the
LoRA adapter improved that metric. Open-ended titles can have low absolute overlap,
so inspect both the metric deltas and the sample titles.

<br><hr>

# Assessment

Use the table above to answer:

- Which overlap metrics improved most after LoRA fine-tuning?
- Do the sample titles become more concise and closer to the reference style?
- Did any questions get worse despite a positive average delta?
- When would you choose LoRA (one-time training) over ICL (larger prompt on every call)?

Notebook 2's Nemotron scores are a useful larger-model reference, but the true control
for this experiment is the base Llama-3.2-3B pass reported here.

<br><hr>

### Wrap-up

You completed the full evaluate → improve loop on a single DGX Spark: deploy a NIM
(nb 00), evaluate it (nb 01/02), and fine-tune with LoRA via NeMo AutoModel (this
notebook) — all locally on one GB10, with open-source tooling.